<a href="https://colab.research.google.com/github/DungeonProgger/IOS-Android/blob/main/Pract3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Описание выборки
Для тестирования был собран сбалансированный мини-датасет из 20 изображений (10 классических и 10 умных часов). Главная особенность выборки — наличие сложных пограничных случаев, призванных проверить устойчивость нейросети:
* **Электронные ретро-часы** (например, Casio F-91W), которые выглядят технологично, но не имеют умной ОС.
* **Гибридные смарт-часы** (например, Withings), оснащенные физическими стрелками и скрытыми датчиками.
* **Умные часы**, дизайн которых полностью мимикрирует под классические хронографы (металлический безель, круглый цифровой циферблат со стрелками).

### 2. Выбор моделей и архитектурный подход
В процессе выполнения работы мы отказались от генеративных VQA-моделей (Visual Question Answering), так как на закрытые вопросы они склонны генерировать синонимы («modern», «standard» или просто «yes») вместо ожидаемых строгих классов.

Были выбраны две модели архитектуры **CLIP (Contrastive Language-Image Pretraining)** от OpenAI разного размера:
1. `openai/clip-vit-base-patch32` (Базовая версия)
2. `openai/clip-vit-large-patch14` (Увеличенная версия)

Архитектура CLIP вычисляет точную математическую вероятность принадлежности картинки к заранее заданным текстовым меткам (в нашем случае `classic watch` и `smartwatch`), что гарантирует 100% предсказуемость формата ответа.

### 3. Логика работы алгоритма
1. **Изоляция окружения:** Выборка читается локально с подключенного Google Drive.
2. **Инициализация Pipelines:** Обе модели загружаются напрямую в RAM виртуальной машины.
3. **Прямой инференс:** Каждое изображение передается в модели вместе со строгим списком `candidate_labels = ["classic watch", "smartwatch"]`.
4. **Сбор и оценка:** Скрипт извлекает наиболее вероятный ответ и вычисляет ключевые метрики классификации (`accuracy`, `precision`, `recall`, `f1-score`) с помощью библиотеки `scikit-learn`.

In [6]:
import pandas as pd
from PIL import Image
from google.colab import drive
from sklearn.metrics import classification_report, accuracy_score
from transformers import pipeline
import warnings

warnings.filterwarnings("ignore")

drive.mount('/content/drive')

#специально предназначенные для классификации мультимодальные модели CLIP
clip_base = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
clip_large = pipeline("zero-shot-image-classification", model="openai/clip-vit-large-patch14")
print("Модели успешно загружены!\n")

DATASET = [
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/1.jpg", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/2.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/3.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/4.jpg", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/5.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/6.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/7.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/8.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/9.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/10.jfif", "label": "classic"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/11.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/12.jpg", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/13.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/14.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/15.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/16.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/17.jpg", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/18.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/19.jfif", "label": "smart"},
    {"file": "/content/drive/MyDrive/Colab Notebooks/Pract/Watches/20.jfif", "label": "smart"}
]

candidate_labels = ["classic watch", "smartwatch"]

results = []
print("Начинаем локальный анализ файлов...")

for idx, item in enumerate(DATASET):
    file_name = item["file"].split('/')[-1]
    print(f"Анализ {idx+1}/{len(DATASET)}: Файл {file_name}...")

    row = {"Файл": file_name, "Истинный_класс": item["label"]}

    try:
        image_obj = Image.open(item["file"]).convert("RGB")

        # Предсказание 1: CLIP Base
        res_base = clip_base(image_obj, candidate_labels=candidate_labels)
        # res_base возвращает список, отсортированный по уверенности модели. Берем первый элемент.
        top_label_base = res_base[0]['label']
        row["Предсказание_CLIP-Base"] = "classic" if "classic" in top_label_base else "smart"

        # Предсказание 2: CLIP Large
        res_large = clip_large(image_obj, candidate_labels=candidate_labels)
        top_label_large = res_large[0]['label']
        row["Предсказание_CLIP-Large"] = "classic" if "classic" in top_label_large else "smart"

    except FileNotFoundError:
        print(f"Файл {file_name} не найден!")
        row["Предсказание_CLIP-Base"] = "error"
        row["Предсказание_CLIP-Large"] = "error"
    except Exception as e:
        print(f"Ошибка обработки {file_name}: {repr(e)}")
        row["Предсказание_CLIP-Base"] = "error"
        row["Предсказание_CLIP-Large"] = "error"

    results.append(row)

df = pd.DataFrame(results)
display(df)

print("\n=== СРАВНЕНИЕ МЕТРИК ДВУХ МОДЕЛЕЙ ===")
y_true = df["Истинный_класс"]
model_names = ["Предсказание_CLIP-Base", "Предсказание_CLIP-Large"]

for m_col in model_names:
    y_pred = df[m_col]

    acc = accuracy_score(y_true, y_pred)

    print(f"\nМодель: {m_col}")
    print(f"Точность (Accuracy): {acc:.2f}")
    print("Отчет о классификации:")
    print(classification_report(y_true, y_pred, zero_division=0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Загрузка моделей Zero-Shot Classification (потребуется пара минут)...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Модели успешно загружены!

Начинаем локальный анализ файлов...
Анализ 1/20: Файл 1.jpg...
Анализ 2/20: Файл 2.jfif...
Анализ 3/20: Файл 3.jfif...
Анализ 4/20: Файл 4.jpg...
Анализ 5/20: Файл 5.jfif...
Анализ 6/20: Файл 6.jfif...
Анализ 7/20: Файл 7.jfif...
Анализ 8/20: Файл 8.jfif...
Анализ 9/20: Файл 9.jfif...
Анализ 10/20: Файл 10.jfif...
Анализ 11/20: Файл 11.jfif...
Анализ 12/20: Файл 12.jpg...
Анализ 13/20: Файл 13.jfif...
Анализ 14/20: Файл 14.jfif...
Анализ 15/20: Файл 15.jfif...
Анализ 16/20: Файл 16.jfif...
Анализ 17/20: Файл 17.jpg...
Анализ 18/20: Файл 18.jfif...
Анализ 19/20: Файл 19.jfif...
Анализ 20/20: Файл 20.jfif...


,Файл,Истинный_класс,Предсказание_CLIP-Base,Предсказание_CLIP-Large
0,1.jpg,classic,classic,classic
1,2.jfif,classic,classic,classic
2,3.jfif,classic,classic,classic
3,4.jpg,classic,classic,classic
4,5.jfif,classic,classic,classic
5,6.jfif,classic,smart,smart
6,7.jfif,classic,classic,classic
7,8.jfif,classic,smart,smart
8,9.jfif,classic,classic,classic
9,10.jfif,classic,classic,classic



=== СРАВНЕНИЕ МЕТРИК ДВУХ МОДЕЛЕЙ ===

Модель: Предсказание_CLIP-Base
Точность (Accuracy): 0.80
Отчет о классификации:
              precision    recall  f1-score   support

     classic       0.80      0.80      0.80        10
       smart       0.80      0.80      0.80        10

    accuracy                           0.80        20
   macro avg       0.80      0.80      0.80        20
weighted avg       0.80      0.80      0.80        20


Модель: Предсказание_CLIP-Large
Точность (Accuracy): 0.85
Отчет о классификации:
              precision    recall  f1-score   support

     classic       0.89      0.80      0.84        10
       smart       0.82      0.90      0.86        10

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.85      0.85      0.85        20



## Анализ результатов и выводы

Обе модели успешно обработали датасет, показав следующие результаты классификации:
* **Модель CLIP-Base** показала общую точность (Accuracy) **80%** (4 ошибки из 20). Метрики распределились равномерно по обоим классам.
* **Модель CLIP-Large** показала точность **85%** (3 ошибки из 20). Увеличенная архитектура лучше улавливает мелкие визуальные детали. Полнота (Recall) для класса смарт-часов достигла 0.90 — крупную модель оказалось сложнее обмануть умными часами с «классическим» дизайном циферблата.